# Ragged Tensors

- TODO: expand bullets
- objective: [graph](./ragged.pdf)

In [1]:
import pathlib as pth
import tensorflow as tf
from datetime import datetime
ks = tf.keras
kl = ks.layers

- load our meta data

In [2]:
vocab = ('x', 'y', '+', '-', '*', '=', ',', ':')
vocab += ('0', '1', '2', '3', '4', '5', '6', '7', '8', '9')

tokens = {k: v for v, k in enumerate(vocab, start=5)}
tokens.update({v: k for k, v in tokens.items()})

SEP = tokens[':']

In [3]:
def paths(ps):
    d = pth.Path('/tmp/q/dataset')
    for i in range(ps.num_shards):
        i = '{:0>4d}'.format(i)
        yield str(d / f'shard_{i}.tfrecords')

@tf.function
def caster(d):
    return {k: tf.cast(v, tf.int32) for k, v in d.items()}

@tf.function
def adapter(d):
    ds = tf.RaggedTensor.from_sparse(d['defs'])
    ss = tf.fill([ds.nrows(), 1], SEP)
    os = tf.RaggedTensor.from_sparse(d['op'])
    x = tf.concat([ds, ss, os], axis=1)
    y = tf.RaggedTensor.from_sparse(d['res'])[:, :1].to_tensor()
    return (x.flat_values, x.row_splits), y

In [4]:
def dset_for(ps):
    ds = tf.data.TFRecordDataset(list(paths(ps))).batch(ps.dim_batch)
    fs = {
        'defs': tf.io.VarLenFeature(tf.int64),
        'op': tf.io.VarLenFeature(tf.int64),
        'res': tf.io.VarLenFeature(tf.int64),
    }
    ds = ds.map(lambda x: tf.io.parse_example(x, fs))
    return ds.map(caster).map(adapter)

In [5]:
class Embed(kl.Layer):
    def __init__(self, ps):
        super().__init__(dtype=tf.float32)
        s = (ps.dim_vocab, ps.dim_hidden)
        self.emb_t = self.add_weight(name='emb_t', shape=s)

    def call(self, x):
        fv, rs = x
        x = tf.RaggedTensor.from_row_splits(fv, rs)
        y = tf.ragged.map_flat_values(tf.nn.embedding_lookup, self.emb_t, x)
        return y

In [6]:
class Reflect(kl.Layer):
    def build(self, shape):
        s = shape[-1]
        self.scale = 1 / (s**0.5)
        self.q_w = self.add_weight(name='q_w', shape=(s, s))
        self.k_w = self.add_weight(name='k_w', shape=(s, s))
        self.v_w = self.add_weight(name='v_w', shape=(s, s))
        return super().build(shape)

    def call(self, x):
        q = x.with_values(tf.einsum('ni,ij->nj', x.flat_values, self.q_w))
        k = x.with_values(tf.einsum('ni,ij->nj', x.flat_values, self.k_w))
        v = x.with_values(tf.einsum('ni,ij->nj', x.flat_values, self.v_w))
        y = tf.einsum('bsi,bzi->bsz', q.to_tensor(), k.to_tensor())
        y = tf.nn.softmax(y * self.scale)
        y = tf.einsum('bsz,bzi->bsi', y, v.to_tensor())
        y = tf.RaggedTensor.from_tensor(y, lengths=x.row_lengths())
        return y

In [7]:
class Expand(kl.Layer):
    def __init__(self, ps):
        super().__init__()
        self.ps = ps

    def call(self, x):
        y = x.to_tensor()
        s = tf.shape(y)[-2]
        y = tf.pad(y, [[0, 0], [0, self.ps.len_max_input - s], [0, 0]])
        return y

In [8]:
def model_for(ps):
    x = [
        ks.Input(shape=(), dtype='int32'),  # , ragged=True)
        ks.Input(shape=(), dtype='int64'),
    ]
    y = Embed(ps)(x)
    y = Reflect()(y)
    y = Expand(ps)(y)
    y = kl.Reshape((ps.len_max_input * ps.dim_hidden, ))(y)
    y = kl.Dense(ps.dim_dense, activation='relu')(y)
    y = kl.Dense(ps.dim_vocab, name='out', activation=None)(y)
    m = ks.Model(inputs=x, outputs=y)
    m.compile(optimizer=ps.optimizer, loss=ps.loss, metrics=[ps.metrics])
    print(m.summary())
    return m

In [9]:
params = dict(
    dim_batch=2,
    dim_dense=150,
    dim_hidden=6,
    dim_vocab=len(vocab) + 5,
    len_max_input=20,
    loss=ks.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=ks.metrics.SparseCategoricalAccuracy(),
    num_shards=2,
    optimizer=ks.optimizers.Adam(),
)

class Params:
    def __init__(self, **kw):
        for k, v in kw.items():
            setattr(self, k, v)

In [11]:
ps = Params(**params)
ds = dset_for(ps)
m = model_for(ps)
ld = datetime.now().strftime('%Y%m%d-%H%M%S')
ld = f'/tmp/q/logs/{ld}'
cs = [ks.callbacks.TensorBoard(log_dir=ld, histogram_freq=1)]
m.fit(ds, callbacks=cs, epochs=10)

W0624 21:38:23.439969 139960225908352 training_utils.py:1444] Expected a shuffled dataset but input dataset `x` is not shuffled. Please invoke `shuffle()` on input dataset.
W0624 21:38:23.617162 139960225908352 summary_ops_v2.py:1110] Model failed to serialize as JSON. Ignoring... Layers with arguments in `__init__` must override `get_config`.


Model: "model_1"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_3 (InputLayer)            [(None,)]            0                                            
__________________________________________________________________________________________________
input_4 (InputLayer)            [(None,)]            0                                            
__________________________________________________________________________________________________
embed_1 (Embed)                 (None, None, 6)      138         input_3[0][0]                    
                                                                 input_4[0][0]                    
__________________________________________________________________________________________________
reflect_1 (Reflect)             (None, None, 6)      108         embed_1[0][0]              

/home/qpix/clone/qnarre-local/.qpy/lib/python3.7/site-packages/tensorflow_core/python/framework/indexed_slices.py:416: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "
/home/qpix/clone/qnarre-local/.qpy/lib/python3.7/site-packages/tensorflow_core/python/framework/indexed_slices.py:416: UserWarning: Converting sparse IndexedSlices to a dense Tensor of unknown shape. This may consume a large amount of memory.
  "Converting sparse IndexedSlices to a dense Tensor of unknown shape. "


1000/1000 [==============================] - 7s 7ms/step - loss: 1.7588 - sparse_categorical_accuracy: 0.5045
Epoch 2/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.6174 - sparse_categorical_accuracy: 0.5210
Epoch 3/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.4897 - sparse_categorical_accuracy: 0.5650
Epoch 4/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.4034 - sparse_categorical_accuracy: 0.5800
Epoch 5/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.3544 - sparse_categorical_accuracy: 0.5810
Epoch 6/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.3174 - sparse_categorical_accuracy: 0.5915
Epoch 7/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.2800 - sparse_categorical_accuracy: 0.5905
Epoch 8/10
1000/1000 [==============================] - 3s 3ms/step - loss: 1.2422 - sparse_categorical_accuracy: 0.5985
Epoch 9/10
1000/1000 [=====================

- with our TensorBoard `callback` in place, the model's `fit` method will generate the standard summaries
- if you haven't run the code, an already generated graph is [here](./ragged.pdf)

In [12]:
%load_ext tensorboard
%tensorboard --logdir /tmp/q/logs

Reusing TensorBoard on port 6007 (pid 20958), started 10:17:40 ago. (Use '!kill 20958' to kill it.)